# Realistic PSMAP response-regime diagnostics
This notebook separates response-only symmetry statements from cloud-, atom-number-, and pixel-dependent operational conclusions. Methodology: [`docs/response_regime_diagnostics.md`](../docs/response_regime_diagnostics.md).

In [ ]:
from pathlib import Path
import subprocess, sys, json
import pandas as pd
from IPython.display import display, Image
REPO = Path.cwd()
if not (REPO / 'helpers').is_dir(): REPO = REPO.parent


## Configuration
The script records the full physical configuration and thresholds in `summary.json`. The maps default to both realistic fine confocal baseline maps.

In [ ]:
CONFIG = {
    'atom_number': 1_000_000,
    'image_bins_per_axis': 24,
    'output_directory': REPO / 'results' / 'response_regime_diagnostics',
    'random_seed': 0,  # retained for extensions; current integration is deterministic
}
CONFIG


## Execute both real maps
This calls the reusable analysis workflow. It uses the established 3.8 s detection time, nominal 100 µm Gaussian widths, a 10 µm/s hidden-mode scale, and configurable regime criteria defined in the runner.

In [ ]:
cmd = [sys.executable, str(REPO/'python-scripts/run_response_regime_diagnostics.py'),
       '--atoms', str(CONFIG['atom_number']), '--bins', str(CONFIG['image_bins_per_axis']),
       '--output', str(CONFIG['output_directory'])]
subprocess.run(cmd, cwd=REPO, check=True)


## Machine-readable classification table

In [ ]:
summary = pd.read_csv(CONFIG['output_directory']/'summary.csv')
display(summary)
with open(CONFIG['output_directory']/'summary.json') as f: recorded_configuration = json.load(f)['configuration']
recorded_configuration


## Required figures
Geometry panels show response phase with ballistic fibres and generator violation. Orbit panels compare original and added-shear phase orbits. Operational panels show count/shape/total phase Fisher information, profiled hidden-mode information, and alpha/kappa boundary scans.

In [ ]:
for name in summary['map']:
    for suffix in ('response_geometry','phase_orbits','operational_diagnostics'):
        display(Image(filename=str(CONFIG['output_directory']/f'{name}_{suffix}.png')))


## Interpretation
Z0 is operationally response-unresolved for the configured hidden mode but lies near the I/II boundary; Z100 resolves that mode and is Regime II. Both maps intrinsically break ballistic-fibre symmetry, yet both have very needle-like total-phase orbits. Their normalized shapes contain nonzero phase information, so count-only is not information-equivalent. Added final-position shear preserves fibre symmetry while producing two phase quadratures; the tabulated kappa threshold marks entry into the configured carrier-like Regime III.

For present maps use nuisance posterior propagation (Z0) or joint/amortized image inference (especially Z100). Direct phase-and-summary in-situ inference becomes appropriate only after phase conditioning is made robust, for example by the scanned shear. Simulator-derived bias is a supervised label, not an experimentally observed per-shot quantity.